# 09 — Forecasting the next BCM value

08 and 08_04 estimate `BCM_GUN` of a shot from **the same shot's** features (a virtual BCM). Here the task is a
**true forecast**: after row `t` has been read out, predict the BCM of row `t + H` (`H` = 1: the next row) from
everything known at that moment, i.e. the rows `≤ t`: their features **and** their BCM readings. Nothing of row
`t + H` is used, neither its BCM nor its features.

**Data:** the datasets of 08 (`results/datasets/`, one file per segment, one row per shot, rows never dropped), with
**segment 8 left out** (bad data). A row holds everything measured for its shot: `extract_features.py` has already
moved the group-B readings (laser meter, PGLA, PKI15A) up by one row. Rows can be 1.2, 2.4 or 3.6 s apart (the
acquisition skips pulses), so `H` counts rows, not machine pulses.

**Comparable with 08 and 08_04:** the target rows are scored on the same test shots (the usable rows in the last
`TEST_FRAC` = 20 % of each segment), early stopping on the last `VAL_FRAC` = 10 % of each segment's training part.

**Models:**
- **baselines** from the BCM history alone: last reading, mean of the last 9 readings (the challenge baseline),
  trailing mean of 50 readings;
- **per-segment tabular models** of 08 (linear, random forest, XGBoost, MLP) on lagged BCM and feature summaries;
- **temporal CNN** of 08_04, one per regime, on windows of the last `L` rows: BCM history only, features only, both;
- **vBCM** (reference, not a forecast): the same-shot models of 08 / 08_04 on the target row's own features. The gap
  to it is what is lost by having to predict one row ahead.

Set the environment variable `SMOKE=1` for a quick end-to-end check (few trees and epochs, one seed): its numbers are
meaningless.

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATASETS = REPO / "results" / "datasets"
SMOKE = os.environ.get("SMOKE") == "1"          # quick end-to-end check only

DROP = {8}                                      # segment 8: bad data
REGIMES = {"low charge": [0, 2, 3, 4], "high, repetitive": [5], "high, non-repetitive": [1, 6, 7]}
REGIME_OF = {s: r for r, segs in REGIMES.items() for s in segs}
TEST_FRAC = 0.2    # as 08: the last 20 % of each segment's shots are the test set
VAL_FRAC = 0.1     # early stopping on the last 10 % of each segment's training part (as 08_04)

H = 1                           # horizon: BCM of row t + H from the rows <= t
LAGS = 5                        # last BCM readings given to the tabular models
LEVELS = (5, 20, 50)            # trailing means of the BCM; the last one is the level the models correct
DROPOUT = 0.2                   # a reading below 0.2 x the trailing median is a dropout: not used as input
GROUP_B = ["lab_laser_power", "PGLA_drop", "PKI15A"]   # moved up one row by extract_features.py
B_LAG = 1                       # forecast inputs: group B one more row back (row t may hold shot t + 1, see below)
HORIZONS = (1, 2, 5) if SMOKE else (1, 2, 5, 10, 20, 50)

N_TREES = 20 if SMOKE else 300
MLP_ITER = 20 if SMOKE else 2000
L = 50                          # CNN window: the origin row and the 49 before it
WIDTH, KERNEL, DILATIONS = 32, 5, (1, 2, 4, 8)   # receptive field 61 rows, as 08_04
SEEDS = (0,) if SMOKE else (0, 1, 2)
BATCH, EPOCHS, PATIENCE, LR, WD = 256, 2 if SMOKE else 60, 8, 1e-3, 1e-4

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(8)
print(f"device {DEVICE}" + (f" ({torch.cuda.get_device_name()})" if DEVICE == "cuda" else "")
      + (" -- SMOKE TEST" if SMOKE else ""))

## Shots and BCM history

Per segment, as 08_04: **usable** = `valid` with a beam on the camera (the scored rows of 08), **test** = the last
`TEST_FRAC` of the segment's shots. Camera features of a repeated image are set missing.

What is known at row `t`, computed causally within each segment (never across a segment boundary):
- **BCM history:** the readings up to `t`, without dropouts (a reading below `DROPOUT` × the median of the previous
  50 is a beam or BCM dropout, ~140 rows in total, mostly segments 3–5: it says nothing about the next shot). The
  last good reading is carried forward.
- **level:** the mean of the last 50 good readings. All models below predict the deviation of BCM(`t + H`) from it.
- **feature history:** the features up to `t`, the last reading carried forward over missing ones. The group-B
  features (laser meter, PGLA, PKI15A) are taken `B_LAG` rows further back: the fixed one-row shift of
  `extract_features.py` does not hold for every row (scripts/README: a meter reading belongs to the previous or the
  same pulse depending on when it was read), so row `t` can hold shot `t + 1`'s reading. The check below shows it.

Targets are not cleaned: a dropout in the test part is scored as in 08 and no forecast can predict it.

In [ ]:
manifest = json.loads((DATASETS / "manifest.json").read_text())
FEATURES = manifest["features"]
CAMERA = [f for f in FEATURES if f.startswith(("BTV_", "SVD_"))]
GROUP = {f: "camera" if f in CAMERA else f for f in FEATURES}         # one missing flag per instrument
SEGMENTS = [info for info in manifest["segments"] if info["segment"] not in DROP]
K = LEVELS[-1]

parts = []
for info in SEGMENTS:
    d = np.load(DATASETS / info["file"])
    p = pd.DataFrame(d["X"], columns=d["features"])
    p["lab_laser_power"] *= 1e6                                       # J -> µJ, see 08
    p.loc[d["btv_frozen"], CAMERA] = np.nan                           # a repeated image says nothing new
    p["BCM_GUN"] = d["y"]
    p["seg"], p["pos"] = info["segment"], np.arange(len(p))
    p["test"] = p["pos"] >= int(len(p) * (1 - TEST_FRAC))
    p["usable"] = d["valid"] & p[FEATURES].notna().all(axis=1).to_numpy()
    parts.append(p)
df = pd.concat(parts, ignore_index=True)
df["regime"] = df["seg"].map(REGIME_OF)
by = df["seg"]                                                        # every rolling / shift stays within a segment
roll = lambda col, w, how: df[col].groupby(by).transform(lambda v: getattr(v.rolling(w, min_periods=min(w, 5)), how)())

# BCM history: dropouts out, last good reading carried forward
median_before = df["BCM_GUN"].groupby(by).transform(lambda v: v.shift(1).rolling(K, min_periods=5).median())
df["dropout"] = df["BCM_GUN"] < DROPOUT * median_before
df["bcm_seen"] = df["BCM_GUN"].where(~df["dropout"])
df["bcm_last"] = df["bcm_seen"].groupby(by).ffill()
for w in LEVELS:
    df[f"bcm_mean{w}"] = roll("bcm_seen", w, "mean")
df["mean of last 9"] = roll("bcm_seen", 9, "mean")
df["base"] = df[f"bcm_mean{K}"]                                       # the level

# tabular inputs, all relative to a trailing level so the trees never have to extrapolate a drift
LAG_COLS = [f"BCM(t-{i}) - level" for i in range(LAGS)]
for i, c in enumerate(LAG_COLS):
    df[c] = df["bcm_last"].groupby(by).shift(i) - df["base"]
MEAN_COLS = [f"BCM mean{w} - level" for w in LEVELS[:-1]]
for w, c in zip(LEVELS, MEAN_COLS):
    df[c] = df[f"bcm_mean{w}"] - df["base"]
HIST = [f"{f} (history)" for f in FEATURES]                          # forecast inputs, group B B_LAG rows back
df[HIST] = df[FEATURES].to_numpy()
df[[f"{f} (history)" for f in GROUP_B]] = df[GROUP_B].groupby(by).shift(B_LAG).to_numpy()
DEV_COLS = [f"{f} - level" for f in FEATURES]
filled = df[HIST].groupby(by).ffill()
df[DEV_COLS] = (filled - filled.groupby(by).transform(lambda v: v.rolling(K, min_periods=5).mean())).to_numpy()
X_COLS = LAG_COLS + MEAN_COLS + DEV_COLS
NEXT = [f"{f} (target row)" for f in FEATURES]                       # for the vBCM reference only

print(f"{len(df)} rows in {len(SEGMENTS)} segments, {int(df['dropout'].sum())} BCM dropouts, "
      f"{len(X_COLS)} tabular inputs: {X_COLS}")

**Alignment check.** Correlation of each feature's deviation from its 50-row trailing mean at row `t` with the BCM's
deviation at row `t + lag`. A feature that measures its own shot peaks at lag 0 (the BCM row shows the BCM's own
autocorrelation). A peak at lag +1 means row `t` often holds shot `t + 1`'s reading: used as a forecast input it would
leak the target. Hence `B_LAG`.

In [ ]:
dev = lambda v: v - v.groupby(by).transform(lambda u: u.rolling(K, min_periods=5).mean())
ry = dev(df["BCM_GUN"])
check = {}
for s in [info["segment"] for info in SEGMENTS if info["shots"] > 1000]:
    m = df["seg"] == s
    for f in FEATURES + ["BCM_GUN"]:
        rf = dev(df[f])[m]
        check[(s, f)] = {lag: rf.corr(ry[m].shift(-lag)) for lag in (-1, 0, 1, 2)}
check = pd.DataFrame(check).T.rename_axis(["segment", "feature"])
check.columns = [f"lag {c:+d}" for c in check.columns]
check.round(2).unstack("segment").swaplevel(axis=1).sort_index(axis=1).style.format("{:.2f}").background_gradient(
    cmap="RdBu_r", vmin=-1, vmax=1)

## Samples for horizon H

One sample per **origin row** `t`: inputs from the rows `≤ t`, target = BCM of row `t + H` of the same segment.
The flags follow the target row, so the scored targets are the test shots of 08 for every `H`:
- **scored:** target row usable and in the test part;
- **val:** the last `VAL_FRAC` of the usable target rows before the test part; **train:** the ones before.

A sample also needs its inputs (the first few rows of a segment have no level yet).

In [ ]:
def samples(h):
    # df + target columns and train / val / scored flags for horizon h, one row per origin row
    S = df.copy()
    gs = S.groupby("seg")
    S["target"] = gs["BCM_GUN"].shift(-h)
    S["dy"] = S["target"] - S["base"]                                  # what the models learn
    t_usable, t_test = gs["usable"].shift(-h).eq(True), gs["test"].shift(-h).eq(True)
    S[NEXT] = gs[FEATURES].shift(-h).to_numpy()
    ok = t_usable & S[X_COLS + ["base"]].notna().all(axis=1)
    S["scored"] = ok & t_test
    before = ok & ~t_test
    k = before.groupby(S["seg"]).cumsum()
    S["val"] = before & (k > (1 - VAL_FRAC) * k.groupby(S["seg"]).transform("max"))
    S["train"] = before & ~S["val"]
    return S


S = samples(H)
S.groupby("seg").agg(regime=("regime", "first"), shots=("pos", "size"), train=("train", "sum"),
                     val=("val", "sum"), test=("scored", "sum"), dropouts=("dropout", "sum"),
                     BCM_mean_nC=("BCM_GUN", "mean")).round(3)

## Baselines and per-segment models

**Baselines**, from the BCM history alone: the last good reading (persistence), the mean of the last 9 (the challenge
baseline; **skill** below is the MAE reduction relative to it) and the level (mean of the last 50).

**Tabular models**, settings of 08, one per segment: linear, random forest, XGBoost, MLP. Inputs at origin `t`: the
last `LAGS` BCM readings, the 5- and 20-reading means, and every feature's deviation from its own 50-row trailing mean,
all relative to the level. Target: BCM(`t + H`) − level; prediction = level + model output. XGBoost stops early on
the val samples, the others are fitted on train + val (the MLP does its own early stopping, see 08).

**vBCM (MLP, target shot):** the MLP of 08 on the target row's own 10 features, no BCM history. A virtual BCM,
not a forecast: the reference for what the same-shot features alone achieve.

In [ ]:
def mlp():
    return TransformedTargetRegressor(
        regressor=make_pipeline(StandardScaler(),
                                MLPRegressor(hidden_layer_sizes=(32, 16), activation="relu", alpha=1e-3,
                                             learning_rate_init=1e-3, max_iter=MLP_ITER, early_stopping=True,
                                             validation_fraction=0.2, n_iter_no_change=30, random_state=0)),
        transformer=StandardScaler())                                  # standardise the target too


TABULAR = {
    "linear": make_pipeline(StandardScaler(), LinearRegression()),
    "random forest": RandomForestRegressor(n_estimators=N_TREES, min_samples_leaf=20, max_features=0.5, n_jobs=-1,
                                           random_state=0),
    "xgboost": XGBRegressor(n_estimators=10 * N_TREES, learning_rate=0.03, max_depth=4, subsample=0.8,
                            colsample_bytree=0.8, min_child_weight=20, early_stopping_rounds=50, eval_metric="mae",
                            n_jobs=8, random_state=0),
    "MLP": mlp(),
}
BASELINES = {"last reading": "bcm_last", "mean of last 9": "mean of last 9", f"level (mean of last {K})": "base"}
VBCM = "vBCM (MLP, target shot)"


def segment_models(S, k, names=tuple(TABULAR), vbcm=True):
    # Baselines and the tabular models on segment k of samples S -> {name: predicted BCM (nC)} on its scored rows
    s = S[S["seg"] == k]
    fit, val, sc = s[s["train"]], s[s["val"]], s[s["scored"]]
    trv = s[s["train"] | s["val"]]
    out = {name: sc[col] for name, col in BASELINES.items()}
    for name in names:
        model = clone(TABULAR[name])
        if name == "xgboost":
            model.fit(fit[X_COLS], fit["dy"], eval_set=[(val[X_COLS], val["dy"])], verbose=False)
        else:
            model.fit(trv[X_COLS], trv["dy"])
        out[name] = pd.Series(sc["base"] + model.predict(sc[X_COLS]), index=sc.index)
    if vbcm:
        out[VBCM] = pd.Series(mlp().fit(trv[NEXT], trv["target"]).predict(sc[NEXT]), index=sc.index)
    return out


preds = {}
for info in SEGMENTS:
    for name, p in segment_models(S, info["segment"]).items():
        preds.setdefault(name, []).append(p)
    print(f"segment {info['segment']} done")
preds = {name: pd.concat(p) for name, p in preds.items()}

## Temporal CNN, one model per regime

The network of 08_04 (four causal dilated 1-D convolutions, receptive field 61 rows, a small head that also reads the
newest step's inputs), on windows of the last `L` rows ending at the origin row `t`. Steps before the segment's first
shot are all-missing: a window never reaches into the previous segment. Input channels per step:
- **features:** the 10 features standardised on the regime's training rows, plus one flag per instrument that is 1
  where its reading is missing (missing value = 0);
- **BCM:** the good reading minus the sample's level, divided by the spread of the target, plus a flag that is 1 on
  dropouts and missing steps. Subtracting the level per window makes the input independent of the segment's charge.

Variants: **BCM only**, **features only** (a forecast from the instruments alone, no BCM at all: then the target is
the BCM itself, standardised, as in 08_04), **BCM + features**. The **vBCM CNN** is 08_04's model (features only, the
window ends at the target row `t + H`, features as in 08_04 without the extra group-B lag): again a reference, not a
forecast.

Training as 08_04: L1 loss, AdamW, early stopping on the val samples, predictions averaged over `SEEDS`.

In [ ]:
class TemporalCNN(torch.nn.Module):
    def __init__(self, c_in, width=WIDTH, kernel=KERNEL, dilations=DILATIONS):
        super().__init__()
        self.pads = [(kernel - 1) * d for d in dilations]
        self.convs = torch.nn.ModuleList(torch.nn.Conv1d(c_in if i == 0 else width, width, kernel, dilation=d)
                                         for i, d in enumerate(dilations))
        self.head = torch.nn.Sequential(torch.nn.Linear(width + c_in, 32), torch.nn.ReLU(), torch.nn.Linear(32, 1))

    def forward(self, x):                                             # (batch, L, channels), newest last
        h = x.transpose(1, 2)
        for conv, pad in zip(self.convs, self.pads):
            h = F.relu(conv(F.pad(h, (pad, 0))))                      # left padding only: causal
        return self.head(torch.cat([h[:, :, -1], x[:, -1]], dim=1)).squeeze(-1)


def step_table(S, rows, inputs, cols):
    # The regime's rows -> step table (rows + 1 all-missing pad row, channels) on DEVICE; BCM channels last
    tr = S.loc[rows, "train"].to_numpy()
    blocks, pad = [], []
    if "features" in inputs:
        X = S.loc[rows, cols].set_axis(FEATURES, axis=1)
        X = (X - X[tr].mean()) / X[tr].std()
        flags = pd.DataFrame({g: X[[f for f in FEATURES if GROUP[f] == g]].isna().any(axis=1)
                              for g in dict.fromkeys(GROUP.values())})
        blocks += [X.fillna(0).to_numpy(), flags.to_numpy()]
        pad += [0.0] * len(FEATURES) + [1.0] * flags.shape[1]
    if "BCM" in inputs:
        b = S.loc[rows, "bcm_seen"]
        blocks += [b.fillna(0).to_numpy()[:, None], b.isna().to_numpy()[:, None]]
        pad += [0.0, 1.0]
    steps = np.hstack(blocks).astype(np.float32)
    return torch.from_numpy(np.vstack([steps, np.float32(pad)])).to(DEVICE)


def windows(S, rows, L, lead=0):
    # rows x L indices into the step table: the window of L rows ending at row + lead, pad row outside the segment
    pos = S.loc[rows, "pos"].to_numpy()
    n = S.loc[rows, "seg"].map(S.groupby("seg").size()).to_numpy()
    step = pos[:, None] + lead - np.arange(L - 1, -1, -1)
    I = np.where((step >= 0) & (step < n[:, None]), np.arange(len(rows))[:, None] + step - pos[:, None], len(rows))
    return torch.from_numpy(I).to(DEVICE)


def fit_one(batch, n_in, Y, fit, val, n, seed):
    torch.manual_seed(seed)
    model = TemporalCNN(n_in).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    predict = lambda b: torch.cat([model(batch(c)) for c in b.split(8192)])
    best, state, wait = np.inf, None, 0
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for b in fit[torch.randperm(len(fit), device=DEVICE)].split(BATCH):
            loss = (model(batch(b)) - Y[b]).abs().mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            v = (predict(val) - Y[val]).abs().mean().item()
        if v < best:
            best, wait, state = v, 0, {k: t.detach().clone() for k, t in model.state_dict().items()}
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    model.load_state_dict(state)
    model.eval()
    with torch.no_grad():
        return predict(torch.arange(n, device=DEVICE)).cpu().numpy(), epoch, best


def temporal_cnn(S, rows, L, inputs, lead=0):
    # The network on windows of L rows ending at row + lead -> predicted BCM of the target row (nC) on the rows
    A, I = step_table(S, rows, inputs, FEATURES if lead else HIST), windows(S, rows, L, lead)
    with_bcm = "BCM" in inputs
    t = S.loc[rows, "target"].fillna(0).to_numpy(np.float32)
    base = S.loc[rows, "base"].fillna(0).to_numpy(np.float32) if with_bcm else np.zeros_like(t)
    tr = S.loc[rows, "train"].to_numpy()
    mu, sd = (t - base)[tr].mean(), (t - base)[tr].std()
    Y = torch.from_numpy((t - base - mu) / sd).to(DEVICE)
    B = torch.from_numpy(base).to(DEVICE)

    def batch(b):
        x = A[I[b]]
        if with_bcm:                                                  # BCM relative to the sample's level
            x[:, :, -2] = (x[:, :, -2] - B[b, None]) / sd * (1 - x[:, :, -1])
        return x

    fit, val = (torch.from_numpy(np.flatnonzero(S.loc[rows, k].to_numpy())).to(DEVICE) for k in ("train", "val"))
    runs = [fit_one(batch, A.shape[1], Y, fit, val, len(rows), seed) for seed in SEEDS]
    print(f"  {'+'.join(inputs)}, L={L}, lead={lead}: epochs {[r[1] for r in runs]}, "
          f"validation MAE {[round(float(r[2] * sd * 1000), 1) for r in runs]} pC")
    return pd.Series(np.mean([r[0] for r in runs], axis=0) * sd + mu + base, index=rows)


CNNS = {"CNN, BCM only": ("BCM",), "CNN, features only": ("features",), "CNN, BCM + features": ("features", "BCM")}
VBCM_CNN = "vBCM (CNN, target shot)"

for rname, segs in REGIMES.items():
    print(rname)
    rows = S.index[S["seg"].isin(segs)]
    for name, inputs in CNNS.items():
        preds.setdefault(name, []).append(temporal_cnn(S, rows, L, inputs))
    preds.setdefault(VBCM_CNN, []).append(temporal_cnn(S, rows, L, ("features",), lead=H))
for name in [*CNNS, VBCM_CNN]:
    preds[name] = pd.concat(preds[name])

## Results, H = 1

Same layout as 08 and 08_04, same test shots. Best per segment in bold, **among the forecasts only** (the two vBCM
columns use the target shot's features and are not forecasts). `skill` = MAE reduction against the mean of the
last 9 readings (%, positive = better). The BCM's own shot-to-shot noise sets a floor no forecast can beat.

In [ ]:
FORECASTS = [*BASELINES, *TABULAR, *CNNS]
MODELS = FORECASTS + [VBCM, VBCM_CNN]
REFERENCE = "mean of last 9"


def summarise(S, preds, names):
    rows, curves = [], {}
    for info in SEGMENTS:
        k = info["segment"]
        i = S.index[S["scored"] & (S["seg"] == k)]
        y = S.loc[i, "target"]
        row = {("", "regime"): REGIME_OF[k], ("", "train"): int((S["train"] | S["val"])[S["seg"] == k].sum()),
               ("", "test"): len(i), ("", "BCM mean (pC)"): 1000 * y.mean(), ("", "BCM std (pC)"): 1000 * y.std()}
        mae = {n: 1000 * mean_absolute_error(y, preds[n][i]) for n in names}
        for n in names:
            row[("MAE (pC)", n)] = mae[n]
        for n in names:
            row[("R2", n)] = r2_score(y, preds[n][i])
        for n in names:
            row[("skill (%)", n)] = 100 * (1 - mae[n] / mae[REFERENCE])
        rows.append(pd.Series(row, name=k))
        curves[k] = (y.to_numpy(), {n: preds[n][i].to_numpy() for n in names})
    summary = pd.DataFrame(rows).rename_axis("segment")
    summary.columns = pd.MultiIndex.from_tuples(summary.columns)
    return summary, curves


def show(summary, block):
    cols = [c for c in summary.columns if c[0] == block]
    best = [c for c in cols if c[1] in FORECASTS]
    st = summary[[c for c in summary.columns if c[0] == ""] + cols].style
    st = st.format("{:.0f}", subset=[("", "train"), ("", "test"), ("", "BCM mean (pC)")]).format(
        "{:.1f}", subset=[("", "BCM std (pC)")]).format("{:.3f}" if block == "R2" else "{:.1f}", subset=cols)
    pick = st.highlight_min if block == "MAE (pC)" else st.highlight_max
    return pick(subset=best, axis=1, props="font-weight: bold")


summary, curves = summarise(S, preds, MODELS)
show(summary, "MAE (pC)")

In [ ]:
show(summary, "skill (%)")

In [ ]:
show(summary, "R2")

Left: the first `ZOOM` scored test shots of each segment, measured and forecast (zoomed in so that a forecast that
only follows the last reading shows as a curve shifted by one shot). Right: forecast vs measured on all test shots.

In [ ]:
ZOOM = 300
COLORS = {"mean of last 9": "#8a8985", "linear": "#2a78d6", "xgboost": "#2e9e5b", "CNN, BCM + features": "#8e5bd0",
          VBCM_CNN: "#e34948"}
fig, axes = plt.subplots(len(curves), 2, figsize=(15, 3.3 * len(curves)), gridspec_kw={"width_ratios": [2.2, 1]},
                         layout="constrained", squeeze=False)
for (ax1, ax2), (s, (y, p)) in zip(axes, sorted(curves.items())):
    shots = np.arange(min(ZOOM, len(y)))
    ax1.plot(shots, y[shots], color="#2b2b29", lw=1.0, label="measured")
    for name, color in COLORS.items():
        ax1.plot(shots, p[name][shots], color=color, lw=0.8, alpha=0.8, label=name,
                 ls=":" if name.startswith("vBCM") else "-")
        ax2.scatter(y, p[name], s=3, color=color, alpha=0.3)
    ax2.plot([y.min(), y.max()], [y.min(), y.max()], color="#2b2b29", lw=0.8)
    run = manifest["segments"][s]["run"].removeprefix("acquisition_")
    ax1.set(ylabel="BCM_GUN (nC)", title=f"segment {s} ({run}, {REGIME_OF[s]}): first {len(shots)} test shots, H = {H}")
    ax2.set(ylabel="forecast (nC)", title="forecast vs measured")
axes[0, 0].legend(fontsize=8, ncols=3)
axes[-1, 0].set_xlabel("test shot (time order, usable shots only)")
axes[-1, 1].set_xlabel("measured (nC)")
plt.show()

## Longer horizons

The baselines, the linear model and XGBoost for `H` in `HORIZONS` (the CNNs are left out: one training per regime,
variant and seed for every `H`). Per regime, the MAE summed over its segments' test shots, relative to the mean of the
last 9 readings at the same `H` (below 1 = better than the challenge baseline). The vBCM does not depend on `H`
(it uses the target shot) and is shown for comparison.

In [ ]:
SWEEP = ["linear", "xgboost"]
errors = []
for h in HORIZONS:
    Sh = samples(h)
    for info in SEGMENTS:
        k = info["segment"]
        p = segment_models(Sh, k, names=SWEEP, vbcm=(h == HORIZONS[0]))
        y = Sh.loc[p[REFERENCE].index, "target"]
        for name, v in p.items():
            errors.append({"H": h, "regime": REGIME_OF[k], "model": name, "abs err (nC)": (v - y).abs().sum()})
    print(f"H = {h} done")
errors = pd.DataFrame(errors).groupby(["regime", "H", "model"])["abs err (nC)"].sum().unstack("model")
ratio = errors.div(errors[REFERENCE], axis=0)                         # MAE / MAE(mean of last 9), per regime and H
display(ratio.round(3))

fig, axes = plt.subplots(1, len(REGIMES), figsize=(15, 3.8), layout="constrained", sharey=True)
LINES = {"last reading": "#d18f00", f"level (mean of last {K})": "#8a8985", "linear": "#2a78d6",
         "xgboost": "#2e9e5b", VBCM: "#e34948"}
for ax, rname in zip(axes, REGIMES):
    for name, color in LINES.items():
        r = ratio.loc[rname, name].dropna()
        if name == VBCM:
            ax.axhline(r.iloc[0], color=color, ls=":", lw=1, label=name)
        else:
            ax.plot(r.index, r.to_numpy(), "o-", color=color, ms=3, lw=1, label=name)
    ax.axhline(1, color="#2b2b29", lw=0.8)
    ax.set(xscale="log", xlabel="horizon H (rows)", title=rname)
axes[0].set_ylabel("MAE / MAE(mean of last 9)")
axes[0].legend(fontsize=8)
plt.show()